# Walkthrough Video → 3D Gaussian Splat

Turn a smartphone walkthrough video of an indoor space into a downloadable **3D Gaussian Splat**
(`.ply` + `.splat`), entirely in Google Colab.

**How it works:** extract frames → recover camera poses with COLMAP (Structure-from-Motion) →
optimize Gaussians with Nerfstudio's [Splatfacto](https://docs.nerf.studio/nerfology/methods/splat.html)
(`gsplat`) → export. No pretrained weights; the splat is fit to your own scene.

> Enable a GPU first: **Runtime → Change runtime type → GPU** (a T4 works; L4/A100 are faster),
> then run the cells top to bottom.

## 1. Verify GPU

In [ ]:
#@title Verify GPU { display-mode: "form" }
import subprocess, sys

try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"], text=True).strip()
    name, mem, driver = [x.strip() for x in out.split(",")]
except Exception:
    print("No GPU detected.")
    print("Enable one via Runtime -> Change runtime type -> GPU, then re-run this cell.")
    raise SystemExit("A GPU is required.")

print(f"GPU:    {name}")
print(f"VRAM:   {mem}")
print(f"Driver: {driver}")

## 2. Settings

Defaults are tuned for a T4 and indoor scenes.

- **quality_preset** — `balanced` (fast) · `high` (recommended) · `max` (most detail).
- **train_iterations** — higher is better but slower; 30k suits a room or apartment.
- **target_frames** — 100–300 for a walkthrough; more frames give better coverage.
- **sfm_tool** — `colmap` (reliable default) or `hloc` (SuperPoint + SuperGlue, better for
  shaky or low-texture footage).

In [ ]:
#@title Settings { display-mode: "form" }
quality_preset   = "high"     #@param ["balanced", "high", "max"]
train_iterations = 30000      #@param {type:"slider", min:5000, max:60000, step:1000}
target_frames    = 200        #@param {type:"slider", min:50, max:500, step:10}
sfm_tool         = "colmap"   #@param ["colmap", "hloc"]
scene_name       = "my_scene" #@param {type:"string"}

import re, os
scene_name = re.sub(r"[^A-Za-z0-9_\-]", "_", scene_name) or "my_scene"
WORK       = "/content/gsplat_project"
VIDEO_DIR  = f"{WORK}/video"
DATA_DIR   = f"{WORK}/processed/{scene_name}"
OUTPUT_DIR = f"{WORK}/outputs"
EXPORT_DIR = f"{WORK}/exports/{scene_name}"
for d in (VIDEO_DIR, DATA_DIR, OUTPUT_DIR, EXPORT_DIR):
    os.makedirs(d, exist_ok=True)

print(f"Scene '{scene_name}': {quality_preset} preset, {train_iterations} iterations, "
      f"{target_frames} frames, SfM = {sfm_tool}.")

## 3. System tools

Installs COLMAP and FFmpeg, and defines the progress helpers used by later cells.

In [ ]:
#@title Install COLMAP + FFmpeg { display-mode: "form" }
import subprocess, sys, os, time, threading, re

def run(cmd, desc=None, log_path=None):
    """Run a command, streaming a compact status line. Raises on failure."""
    if desc:
        print(f"-> {desc}")
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    tail, logf = [], open(log_path, "w") if log_path else None
    for line in proc.stdout:
        if logf:
            logf.write(line)
        tail.append(line.rstrip()); tail = tail[-400:]
        sys.stdout.write("\r   " + line.rstrip()[:100].ljust(102)); sys.stdout.flush()
    proc.wait()
    if logf:
        logf.close()
    sys.stdout.write("\r" + " " * 106 + "\r"); sys.stdout.flush()
    if proc.returncode != 0:
        print(f"   failed: {desc or cmd}")
        for l in tail[-25:]:
            print("   " + l)
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {cmd}")
    print("   done")
    return "\n".join(tail)

def gpu_stat():
    try:
        q = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,noheader,nounits"], text=True).strip().splitlines()[0]
        u, used, tot = [int(x) for x in q.split(",")]
        return u, used / 1024.0, tot / 1024.0
    except Exception:
        return None

def run_live(cmd, log_path, total_steps=None, stage_map=None):
    """Run a long command with a live progress bar (total_steps) or a stage
    heartbeat (stage_map). Saves full output to log_path; returns (rc, tail)."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    st = {"step": 0, "stage": "starting", "t0": time.time(), "alive": True}
    seen, tail = set(), []

    def stamp(name):
        mm, ss = divmod(int(time.time() - st["t0"]), 60)
        sys.stdout.write("\r" + " " * 102 + "\r")
        print(f"  [{mm}:{ss:02d}] {name}")

    def heartbeat():
        while st["alive"]:
            mm, ss = divmod(int(time.time() - st["t0"]), 60)
            g = gpu_stat()
            gs = f"GPU {g[0]:>3d}% {g[1]:.1f}/{g[2]:.0f}GB" if g else "GPU n/a"
            if total_steps and st["step"] > 0:
                pct = min(st["step"] / total_steps, 1.0); fill = int(pct * 24)
                line = (f"[{'#' * fill}{'.' * (24 - fill)}] {pct * 100:5.1f}% "
                        f"{st['step']}/{total_steps} | {gs} | {mm}:{ss:02d}")
            else:
                line = f"{st['stage']:<24} | {gs} | {mm}:{ss:02d}"
            sys.stdout.write("\r" + line[:100].ljust(102)); sys.stdout.flush()
            time.sleep(4)

    hb = threading.Thread(target=heartbeat, daemon=True); hb.start()
    step_re = re.compile(r"(\d+)\s*\(\s*[\d.]+\s*%\)")
    with open(log_path, "w") as lf:
        for line in proc.stdout:
            lf.write(line); s = line.rstrip(); tail.append(s); tail = tail[-500:]
            if stage_map:
                low = s.lower()
                for kw, name in stage_map.items():
                    if kw in low:
                        st["stage"] = name
                        if name not in seen:
                            seen.add(name); stamp(name)
            if total_steps:
                m = step_re.search(s)
                if m and int(m.group(1)) <= total_steps:
                    if "loop" not in seen:
                        seen.add("loop"); stamp("training started")
                    st["step"] = int(m.group(1))
            if any(k in s for k in ("Traceback", "CUDA out of memory", "Killed",
                                    "core dumped")):
                sys.stdout.write("\r" + " " * 102 + "\r"); print("  " + s[:140])
    proc.wait(); st["alive"] = False; hb.join(timeout=5)
    sys.stdout.write("\r" + " " * 102 + "\r"); sys.stdout.flush()
    return proc.returncode, "\n".join(tail[-45:])

run("apt-get -qq update", "Updating apt")
run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y colmap ffmpeg xvfb",
    "Installing COLMAP + FFmpeg")
print("System tools ready.")

## 4. Install Nerfstudio + gsplat

Installs Nerfstudio (from `main`) and a **prebuilt gsplat wheel** with its CUDA extension compiled in,
so gsplat loads it directly instead of compiling at runtime.

The wheel is built for this stack (Python 3.12, the torch `nerfstudio@main` installs — currently
torch 2.11.0 + CUDA 12.8, T4 / `sm_75`). The cell checks that torch is CUDA-enabled before installing
the wheel so the ABI matches. To rebuild the wheel, see the recipe at the end of the notebook.

In [ ]:
#@title Install Nerfstudio + gsplat { display-mode: "form" }
import os, sys, subprocess

# Arch + job cap, in case a torch/ABI mismatch forces gsplat to JIT-compile.
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["MAX_JOBS"] = "2"

nerfstudio_ref   = "main"  #@param {type:"string"}
gsplat_wheel_url = "https://github.com/JittoJoseph/GSplat-build/releases/download/v2/gsplat-1.4.0-cp312-cp312-linux_x86_64.whl"  #@param {type:"string"}

print(f"Installing nerfstudio @ {nerfstudio_ref} (~6-10 min).\n")
run(f"pip -q install 'git+https://github.com/nerfstudio-project/nerfstudio.git@{nerfstudio_ref}'",
    "Installing nerfstudio + gsplat", log_path="/content/install_nerfstudio.log")

# The wheel's ABI must match the torch nerfstudio installed, and needs a CUDA build.
torch_info = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__, torch.version.cuda or 'cpu')"],
    capture_output=True, text=True).stdout.strip()
print("torch:", torch_info)
if torch_info.endswith("cpu"):
    raise RuntimeError("torch has no CUDA build , enable a GPU runtime and re-run.")

run(f"pip -q install --force-reinstall --no-deps '{gsplat_wheel_url}'",
    "Installing prebuilt gsplat wheel")

_check = r"""
import warnings; warnings.filterwarnings('ignore')
import torch, gsplat
try:
    from gsplat import csrc  # noqa: F401
    mode = 'precompiled wheel'
except ImportError:
    mode = 'JIT-compiled'
d = 'cuda'; N = 64
means = torch.rand(N, 3, device=d); quats = torch.randn(N, 4, device=d)
scales = torch.rand(N, 3, device=d) * 0.01; opac = torch.rand(N, device=d)
colors = torch.rand(N, 3, device=d); viewmats = torch.eye(4, device=d)[None]
Ks = torch.tensor([[[100., 0, 32], [0, 100, 32], [0, 0, 1]]], device=d)
gsplat.rasterization(means, quats, scales, opac, colors, viewmats, Ks, 64, 64)
torch.cuda.synchronize()
print('READY', mode)
"""
with open("/content/_gsplat_check.py", "w") as f:
    f.write(_check)
out = run("python /content/_gsplat_check.py", "Verifying gsplat", log_path="/content/gsplat_check.log")
if "READY" not in out:
    raise RuntimeError("gsplat CUDA extension unavailable. See /content/gsplat_check.log")

if subprocess.run(["ns-train", "--help"], capture_output=True).returncode != 0:
    raise RuntimeError("ns-train not available. See /content/install_nerfstudio.log")
print(f"Nerfstudio ready | gsplat: {'precompiled wheel' if 'precompiled wheel' in out else 'JIT-compiled'}.")

## 5. Provide your video

Choose one `input_method`:

- **`upload`** — a *Choose Files* button appears; select the video.
- **`google_drive_link`** — paste a Drive share link (shared "Anyone with the link").
- **`direct_url_or_path`** — a direct `https://…` video URL, or a local/Drive-mounted file path.

Supported types: `.mp4 .mov .m4v .avi .mkv .webm`. Capture tip: move slowly, keep good lighting,
overlap your views, 30–90 s.

In [ ]:
#@title Provide video { display-mode: "form" }
import os, sys, shutil, subprocess, json
input_method = "upload"  #@param ["upload", "google_drive_link", "direct_url_or_path"]
#@markdown Paste here for google_drive_link / direct_url_or_path (leave blank for upload):
link_or_url = ""  #@param {type:"string"}

# To use a Google Drive path: uncomment, run, then pick direct_url_or_path and set the path.
# from google.colab import drive; drive.mount('/content/drive')

VIDEO_EXTS = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")

if input_method == "upload":
    from google.colab import files
    print("Choose your walkthrough video...\n")
    up = files.upload()
    if not up:
        raise RuntimeError("No file selected. Re-run and choose a video.")
    fn = list(up.keys())[0]
    if not fn.lower().endswith(VIDEO_EXTS):
        raise ValueError(f"'{fn}' is not a recognized video type {VIDEO_EXTS}.")
    VIDEO_PATH = os.path.join(VIDEO_DIR, fn)
    open(VIDEO_PATH, "wb").write(up[fn])
else:
    src = link_or_url.strip()
    if not src:
        raise ValueError("link_or_url is empty (or set input_method='upload').")
    if input_method == "google_drive_link" or "drive.google.com" in src or "docs.google.com" in src:
        try:
            import gdown
        except Exception:
            subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gdown"]); import gdown
        print("Downloading from Google Drive...\n")
        got = gdown.download(url=src, output=os.path.join(VIDEO_DIR, "walkthrough_input"),
                             quiet=False, fuzzy=True)
        if not got or not os.path.isfile(got):
            raise RuntimeError("Download failed. Ensure the link is shared 'Anyone with the link' "
                               "and points to a single file.")
        VIDEO_PATH = got
    elif src.startswith(("http://", "https://")):
        ext = os.path.splitext(src.split("?")[0])[1].lower() or ".mp4"
        VIDEO_PATH = os.path.join(VIDEO_DIR, "walkthrough_input" + ext)
        run(f"wget -q --show-progress -O '{VIDEO_PATH}' '{src}'", "Downloading video")
        if os.path.getsize(VIDEO_PATH) < 1024:
            raise RuntimeError("Downloaded file is empty; the URL may not be a direct video link.")
    elif os.path.isfile(src):
        VIDEO_PATH = os.path.join(VIDEO_DIR, os.path.basename(src)); shutil.copy(src, VIDEO_PATH)
    else:
        raise FileNotFoundError(f"'{src}' is neither a URL nor an existing file path.")

probe = subprocess.run(
    "ffprobe -v error -select_streams v:0 -show_entries "
    f"stream=width,height:format=duration -of json '{VIDEO_PATH}'",
    shell=True, capture_output=True, text=True)
print(f"Video ready: {os.path.basename(VIDEO_PATH)} ({os.path.getsize(VIDEO_PATH) / 1e6:.1f} MB)")
try:
    info = json.loads(probe.stdout)
    stream = info["streams"][0]; dur = float(info["format"]["duration"])
    print(f"  {stream['width']}x{stream['height']}, {dur:.1f} s")
    if dur < 5:
        print("  Warning: very short (<5 s); reconstruction may be poor.")
except Exception:
    pass

## 6. Recover camera poses (Structure-from-Motion)

`ns-process-data` extracts frames and runs COLMAP to recover each frame's camera pose and a sparse
point cloud (used to initialize the Gaussians). This is usually the slowest stage. COLMAP runs on
CPU for reliability on headless Colab.

In [ ]:
#@title Run SfM { display-mode: "form" }
import os, json, glob, shutil
if os.path.isdir(DATA_DIR) and os.listdir(DATA_DIR):
    shutil.rmtree(DATA_DIR); os.makedirs(DATA_DIR, exist_ok=True)

# COLMAP GPU SIFT is unreliable on headless Colab, so run it on CPU (--no-gpu); hloc keeps the GPU.
match = "sequential" if sfm_tool == "colmap" else "exhaustive"
gpu_flag = "--no-gpu" if sfm_tool == "colmap" else ""
cmd = (f"xvfb-run -a ns-process-data video --data '{VIDEO_PATH}' --output-dir '{DATA_DIR}' "
       f"--num-frames-target {target_frames} --sfm-tool {sfm_tool} "
       f"--matching-method {match} {gpu_flag}").strip()

stage_map = {
    "extract": "extracting frames", "feature": "extracting features",
    "matching": "matching features", "match": "matching features",
    "reconstruct": "sparse mapping", "mapper": "sparse mapping",
    "bundle": "bundle adjustment", "undistort": "undistorting", "refine": "refining poses",
}
print("Running Structure-from-Motion. Full log: /content/ns_process.log\n")
rc, tail = run_live(cmd, "/content/ns_process.log", stage_map=stage_map)
if rc != 0:
    print("\n--- log tail ---\n", tail)
    raise RuntimeError("SfM failed. Try sfm_tool='hloc', or recapture with more overlap.")

frames = json.load(open(os.path.join(DATA_DIR, "transforms.json"))).get("frames", []) \
    if os.path.isfile(os.path.join(DATA_DIR, "transforms.json")) else []
if not frames:
    raise RuntimeError("SfM produced no valid poses. See /content/ns_process.log")
images = len(glob.glob(os.path.join(DATA_DIR, "images", "*")))
print(f"\nRegistered {len(frames)} / {images} frames.")
if len(frames) < 20:
    print("Warning: fewer than 20 frames registered; quality will be low. "
          "Try sfm_tool='hloc' or recapture.")
else:
    print("Poses look good. Ready to train.")

## 7. Train the Gaussian Splat

Optimizes the Gaussians against your images (no pretrained weights).

- **`high`/`max`** use `splatfacto-mcmc` with a fixed Gaussian budget → predictable VRAM and clean,
  floater-free surfaces (ideal for walls and floors). **`balanced`** uses `splatfacto`.
- Frames are cached in GPU memory so the GPU never waits on data loading.
- Camera poses are refined during training, and a bilateral grid corrects for phone auto-exposure.

A live bar shows step %, GPU utilization, and elapsed time.

In [ ]:
#@title Train { display-mode: "form" }
import os, glob
from PIL import Image

os.environ["TORCHDYNAMO_DISABLE"] = "1"   # run get_viewmat eagerly, skipping a small compile
os.environ["TORCH_COMPILE_DISABLE"] = "1"

cache_mode = "gpu"  #@param ["gpu", "cpu"]
#@markdown `gpu` caches frames in VRAM for fastest training. Switch to `cpu` only if
#@markdown you have very many high-resolution frames and hit a VRAM limit.

BUDGET = {"balanced": 1_000_000, "high": 1_800_000, "max": 3_000_000}
gs_cap = BUDGET[quality_preset]
method = "splatfacto" if quality_preset == "balanced" else "splatfacto-mcmc"

imgs = sorted(glob.glob(os.path.join(DATA_DIR, "images", "*")))
if not imgs:
    raise RuntimeError("No images found. Re-run the SfM cell first.")
W, H = Image.open(imgs[0]).size
factor = 1
while max(W, H) / factor > 1600:          # cap the long side ~1600 px (quality plateau)
    factor *= 2

common = [
    f"--data '{DATA_DIR}'", f"--output-dir '{OUTPUT_DIR}'",
    f"--max-num-iterations {train_iterations}",
    "--viewer.quit-on-train-completion True", "--vis tensorboard",
    f"--experiment-name {scene_name}",
    "--pipeline.model.camera-optimizer.mode SO3xR3",
    "--pipeline.model.use-bilateral-grid True",
    "--pipeline.model.rasterize-mode classic",
]
common += [f"--pipeline.datamanager.cache-images {cache_mode}",
           "--pipeline.datamanager.cache-images-type uint8"]

def model_flags(cap):
    if method == "splatfacto-mcmc":
        return [f"--pipeline.model.max-gs-num {cap}"]
    return ["--pipeline.model.use-scale-regularization True"]

def build(cap):
    return (f"ns-train {method} " + " ".join(common + model_flags(cap)) +
            f" nerfstudio-data --data '{DATA_DIR}' --downscale-factor {factor}")

stages = {"caching / undistorting train": "loading images"}
label = method + (f" ({gs_cap / 1e6:.1f}M gaussians)" if method == "splatfacto-mcmc" else "")
print(f"Training: {quality_preset} preset, {label}, {train_iterations} iterations "
      f"at {W // factor}x{H // factor}.\n")
rc, tail = run_live(build(gs_cap), "/content/ns_train.log",
                    total_steps=train_iterations, stage_map=stages)

if rc != 0 and "out of memory" in tail.lower() and method == "splatfacto-mcmc":
    cap = int(gs_cap * 0.55)
    print(f"\nGPU out of memory. Retrying with ~{cap / 1e6:.1f}M gaussians...\n")
    rc, tail = run_live(build(cap), "/content/ns_train.log",
                        total_steps=train_iterations, stage_map=stages)

if rc != 0:
    print("\n--- log tail ---\n", tail)
    raise RuntimeError("Training failed. See /content/ns_train.log")

cfgs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "config.yml"), recursive=True),
              key=os.path.getmtime)
if not cfgs:
    raise RuntimeError("Training finished but no config.yml was found.")
CONFIG_YML = cfgs[-1]
print("\nTraining complete.")

## 8. Export

Exports a lossless **`.ply`** (best quality — works in SuperSplat, PlayCanvas, Three.js) and a
compact **`.splat`** for lightweight web viewers.

In [ ]:
#@title Export .ply + .splat { display-mode: "form" }
import os, sys, glob, subprocess, numpy as np

# nerfstudio ships open3d but not plyfile, which we need to read the Gaussian fields.
try:
    import plyfile  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "plyfile"])

# PyTorch 2.6+ defaults torch.load(weights_only=True), which rejects the numpy objects in a
# nerfstudio checkpoint, so we patch it to load our own trusted checkpoint.
export_py = r"""
import sys, warnings, torch
warnings.filterwarnings("ignore")
_orig_load = torch.load
torch.load = lambda *a, **k: _orig_load(*a, **{**k, "weights_only": False})
cfg, out = sys.argv[1], sys.argv[2]
sys.argv = ["ns-export", "gaussian-splat", "--load-config", cfg, "--output-dir", out]
from nerfstudio.scripts.exporter import entrypoint
entrypoint()
"""
with open("/content/_export_splat.py", "w") as f:
    f.write(export_py)
run(f"TORCHDYNAMO_DISABLE=1 python /content/_export_splat.py '{CONFIG_YML}' '{EXPORT_DIR}'",
    "Exporting .ply", log_path="/content/ns_export.log")

plys = glob.glob(os.path.join(EXPORT_DIR, "*.ply"))
if not plys:
    raise RuntimeError("No .ply produced. See /content/ns_export.log")
PLY_PATH = plys[0]
print(f".ply:   {PLY_PATH}  ({os.path.getsize(PLY_PATH) / 1e6:.1f} MB)")

def ply_to_splat(ply_path, splat_path):
    # .splat record (32 B): position(3 f32) scale(3 f32) rgba(4 u8) rotation-quat(4 u8),
    # written most-important-first so streaming viewers show the best splats early.
    from plyfile import PlyData
    v = PlyData.read(ply_path)["vertex"]
    xyz = np.stack([v["x"], v["y"], v["z"]], 1).astype(np.float32)
    scales = np.exp(np.stack([v["scale_0"], v["scale_1"], v["scale_2"]], 1)).astype(np.float32)
    rots = np.stack([v["rot_0"], v["rot_1"], v["rot_2"], v["rot_3"]], 1).astype(np.float32)
    rots /= np.linalg.norm(rots, axis=1, keepdims=True) + 1e-9
    c0 = 0.28209479177387814
    color = np.clip(0.5 + c0 * np.stack([v["f_dc_0"], v["f_dc_1"], v["f_dc_2"]], 1), 0, 1)
    opacity = 1 / (1 + np.exp(-np.asarray(v["opacity"], dtype=np.float32)))
    rgba = np.clip(np.concatenate([color, opacity[:, None]], 1) * 255, 0, 255).astype(np.uint8)
    quat = np.clip(rots * 128 + 128, 0, 255).astype(np.uint8)
    order = np.argsort(-(opacity * np.prod(scales, 1)))
    rec = np.zeros(len(order), dtype=[("xyz", "<f4", 3), ("scale", "<f4", 3),
                                      ("rgba", "u1", 4), ("rot", "u1", 4)])
    rec["xyz"] = xyz[order]; rec["scale"] = scales[order]
    rec["rgba"] = rgba[order]; rec["rot"] = quat[order]
    rec.tofile(splat_path)
    return len(order)

SPLAT_PATH = os.path.join(EXPORT_DIR, f"{scene_name}.splat")
try:
    n = ply_to_splat(PLY_PATH, SPLAT_PATH)
    print(f".splat: {SPLAT_PATH}  ({n:,} gaussians, {os.path.getsize(SPLAT_PATH) / 1e6:.1f} MB)")
except Exception as e:
    SPLAT_PATH = None
    print(f"\nWARNING: .splat conversion failed ({e}). The .ply is still available; "
          "re-run this cell to retry the .splat.")

## 9. Download

Bundles the results into a zip and downloads it. To view instantly, open
[superspl.at/editor](https://superspl.at/editor) and drag in the `.ply`.

In [ ]:
#@title Download { display-mode: "form" }
import os, zipfile
zip_path = os.path.join(WORK, f"{scene_name}_gaussian_splat.zip")
items = [p for p in (PLY_PATH, SPLAT_PATH) if p and os.path.isfile(p)]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in items:
        z.write(p, arcname=os.path.basename(p))
    z.writestr("README.txt",
               f"Scene: {scene_name}\n"
               f"Nerfstudio Splatfacto ({quality_preset}, {train_iterations} iters, sfm={sfm_tool})\n"
               ".ply = full quality (recommended); .splat = compact web view.\n"
               "View: https://superspl.at/editor\n")
for p in items:
    print(f"  {os.path.basename(p):36s} {os.path.getsize(p) / 1e6:7.1f} MB")
print(f"\nZip: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download from the Files panel:", zip_path)

## Troubleshooting

| Issue | Fix |
|---|---|
| No GPU detected | Runtime → Change runtime type → GPU, then re-run. |
| gsplat reports `JIT-compiled` (cell 4) | Wheel ABI doesn't match the installed torch; rebuild the wheel for the current torch (recipe below). |
| Few or no frames registered (cell 6) | Set `sfm_tool = 'hloc'`; recapture with slower motion and more overlap. |
| GPU out of memory (cell 7) | Use the `balanced` preset or fewer `target_frames`, or a larger GPU (L4/A100). |
| Noisy or floaty result | Increase `train_iterations`, use `high`/`max`, capture more overlapping views. |

**Higher indoor geometry:** [DN-Splatter](https://github.com/maturk/dn-splatter) adds depth and normal
priors and reuses the dataset produced in cell 6.

### Building the prebuilt gsplat wheel (one-time)

Cell 4 installs a gsplat wheel with the CUDA extension compiled in, so it never JIT-compiles. Build
that wheel once, **on a T4 GPU runtime** (so torch is CUDA-enabled), and host it. gsplat has no
`pyproject.toml`, imports torch in `setup.py`, and its GLM dependency is a git submodule missing from
the PyPI sdist — so build from the repo with submodules, in place:

```bash
pip install 'git+https://github.com/nerfstudio-project/nerfstudio.git@main'   # matching torch
python -c "import torch; assert torch.version.cuda, 'CPU-only torch: use a GPU runtime'"
git clone --recursive https://github.com/nerfstudio-project/gsplat.git /content/gsplat
cd /content/gsplat && git checkout v1.4.0 && git submodule update --init --recursive
# Build in place (one command) so nvcc finds the glm headers:
cd /content/gsplat && MAX_JOBS=2 TORCH_CUDA_ARCH_LIST=7.5 python setup.py bdist_wheel
ls /content/gsplat/dist/   # -> gsplat-1.4.0-cp312-...whl  (host it, set gsplat_wheel_url)
```

**Credits:** Nerfstudio · gsplat · 3D Gaussian Splatting (Kerbl et al., SIGGRAPH 2023) · COLMAP ·
DN-Splatter (Turkulainen et al., WACV 2025).